In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy

import albumentations as A
import mlflow
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

from mermaidseg.datasets.dataset import CoralNetDataset, CombinedCoralDataset, MermaidDataset
from mermaidseg.io import get_parser, setup_config, update_config_with_args
from mermaidseg.logger import Logger
from mermaidseg.model.eval import EvaluatorSemanticSegmentation
from mermaidseg.model.meta import MetaModel
from mermaidseg.model.train import train_model

In [ ]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

## Set MLFlow App

* Set in notebook or as environment variable
* Verify URI after config setup (before training)

In [ ]:
if not os.getenv("MLFLOW_TRACKING_URI"):
    os.environ["MLFLOW_TRACKING_URI"] = "arn:aws:sagemaker:{region}:12345678:mlflow-app/app-ABCDEFG"

# 1. Setup

In [ ]:
device_count = torch.cuda.device_count()
for i in range(device_count):
    print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## Config

In [ ]:
cfg = setup_config(
    config_path="../configs/linear-dinov3-base.yaml",
    config_base_path="../configs/combined_mermaid_coralnet.yaml",
)

args_input = "--run-name=combined_base_dinov3 --log-epochs=5"
args_input = args_input.split(" ")

parser = get_parser()
args = parser.parse_args(args_input)

cfg = update_config_with_args(cfg, args)
cfg_logger = copy.deepcopy(cfg)

# 2. Dataset

In [ ]:
transforms = {}
for split in cfg.augmentation:
    transforms[split] = A.Compose(
        [
            getattr(A, transform_name)(**transform_params)
            for transform_name, transform_params in cfg.augmentation[split].items()
        ]
    )

In [ ]:
batch_size = cfg.data.pop("batch_size", 8)
class_subset = cfg.data.class_subset
padding = cfg.data.padding

In [ ]:
mermaid_full = MermaidDataset(
    transform=transforms["train"],
    class_subset=class_subset,
    padding=padding,
)

total_size = len(mermaid_full)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(seed)
mermaid_train, mermaid_val, mermaid_test = random_split(
    mermaid_full, [train_size, val_size, test_size], generator=generator
)

print(f"MERMAID splits: train={train_size}, val={val_size}, test={test_size}")

In [ ]:
def _load_sources(path: str) -> list:
    return pd.read_csv(path, header=0).values.flatten().tolist()

coralnet_train = CoralNetDataset(
    transform=transforms["train"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/run1_train_sources.csv"),
)
coralnet_val = CoralNetDataset(
    transform=transforms["val"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/run1_val_sources.csv"),
)
coralnet_test = CoralNetDataset(
    transform=transforms["test"],
    class_subset=class_subset,
    padding=padding,
    whitelist_sources=_load_sources("../configs/run1_test_sources.csv"),
)

print(f"CoralNet splits: train={len(coralnet_train)}, val={len(coralnet_val)}, test={len(coralnet_test)}")

In [ ]:
MERMAID_CAP = 8000
CORALNET_CAP = 8000

def _subsample(dataset, cap, generator):
    if len(dataset) <= cap:
        return dataset
    return random_split(dataset, [cap, len(dataset) - cap], generator=generator)[0]

sub_generator = torch.Generator().manual_seed(seed)

# Subsample MERMAID splits proportionally (preserves ~70/15/15 ratio)
mermaid_cap_train = int(MERMAID_CAP * 0.70)
mermaid_cap_val   = int(MERMAID_CAP * 0.15)
mermaid_cap_test  = MERMAID_CAP - mermaid_cap_train - mermaid_cap_val
mermaid_train = _subsample(mermaid_train, mermaid_cap_train, sub_generator)
mermaid_val   = _subsample(mermaid_val,   mermaid_cap_val,   sub_generator)
mermaid_test  = _subsample(mermaid_test,  mermaid_cap_test,  sub_generator)

# Subsample CoralNet within source-split datasets proportionally
coralnet_cap_train = int(CORALNET_CAP * 0.70)
coralnet_cap_val   = int(CORALNET_CAP * 0.15)
coralnet_cap_test  = CORALNET_CAP - coralnet_cap_train - coralnet_cap_val
coralnet_train = _subsample(coralnet_train, coralnet_cap_train, sub_generator)
coralnet_val   = _subsample(coralnet_val,   coralnet_cap_val,   sub_generator)
coralnet_test  = _subsample(coralnet_test,  coralnet_cap_test,  sub_generator)

print(f"MERMAID  (capped): train={len(mermaid_train)}, val={len(mermaid_val)}, test={len(mermaid_test)}")
print(f"CoralNet (capped): train={len(coralnet_train)}, val={len(coralnet_val)}, test={len(coralnet_test)}")

In [ ]:
combined_train = CombinedCoralDataset([mermaid_train, coralnet_train])
combined_val = CombinedCoralDataset([mermaid_val, coralnet_val])
combined_test = CombinedCoralDataset([mermaid_test, coralnet_test])

print(f"Combined splits: train={len(combined_train)}, val={len(combined_val)}, test={len(combined_test)}")

In [ ]:
train_loader = DataLoader(
    combined_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_train.collate_fn,
)
val_loader = DataLoader(
    combined_val,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_val.collate_fn,
)
test_loader = DataLoader(
    combined_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    drop_last=True,
    collate_fn=combined_test.collate_fn,
)

In [ ]:
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
combined_train.num_classes

# 3. Model

In [ ]:
meta_model = MetaModel(
    run_name=cfg.run_name,
    num_classes=combined_train.num_classes,
    device=device,
    model_kwargs=cfg.model,
    training_kwargs=cfg.training,
)

evaluator = EvaluatorSemanticSegmentation(
    num_classes=combined_train.num_classes,
    device=device,
)

: 

In [ ]:
cfg.logger.experiment_name = "mermaid_combined"
cfg_logger.logger.experiment_name = "mermaid_combined"

In [ ]:
logger = Logger(
    config=cfg_logger,
    meta_model=meta_model,
    log_epochs=cfg.logger.log_epochs,
    log_checkpoint=2,
    checkpoint_dir=".",
    enable_mlflow=True,
    enable_wandb=False,
    id2label=combined_train.id2label,
)

In [ ]:
tracking_uri = mlflow.get_tracking_uri()
print(f"Tracking URI: {tracking_uri}")

In [ ]:
logger.log_datasets(combined_train, context="training")

if mlflow.active_run():
    mlflow.log_params({
        "data/mermaid_total_size": total_size,
        "data/mermaid_train_size": train_size,
        "data/mermaid_val_size": val_size,
        "data/mermaid_test_size": test_size,
        "data/combined_train_size": len(combined_train),
        "data/combined_val_size": len(combined_val),
        "data/combined_test_size": len(combined_test),
        "data/seed": seed,
    })

In [ ]:
train_model(meta_model, evaluator, train_loader, val_loader, logger=logger)